# Customer Churn Prediction using Logistic Regression

## Objective

Build a complete customer churn prediction pipeline using Logistic Regression.

The project demonstrates:

- Data cleaning
- Target encoding
- Numerical feature scaling
- Categorical feature encoding
- ColumnTransformer
- Pipeline
- Stratified train-test splitting
- Logistic Regression with balanced class weights
- Prediction on unseen test data

# Problem Statement

A telecom company wants to predict whether a customer is likely to leave its service.

The dataset contains customer information such as:

- Tenure
- Monthly Charges
- Total Charges
- Contract type
- Internet service
- Payment method

The target variable is `Churn`.

The objective is to prepare the data correctly and train a baseline Logistic Regression model.

# Dataset

The sample dataset contains 11 customer records.

Target:

- `Yes` → 1 → Customer churned
- `No` → 0 → Customer did not churn

There is also one missing `TotalCharges` value.

The missing value will be converted to `NaN` using `pd.to_numeric(errors="coerce")` and then replaced with `0`.

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [3]:
# Load the Dataset

# The dataset is provided as an inlined CSV string.

# We use `StringIO` so that Pandas can read the string as if it were a CSV file.

from io import StringIO

sample_csv = """customerID,tenure,MonthlyCharges,TotalCharges,Contract,InternetService,PaymentMethod,Churn
C1001,2,70.35,140.70,Month-to-month,Fiber optic,Electronic check,Yes
C1002,34,56.95,1889.50,One year,DSL,Mailed check,No
C1003,2,53.85,108.15,Month-to-month,DSL,Mailed check,Yes
C1004,45,42.30,1840.75,One year,DSL,Bank transfer (automatic),No
C1005,8,99.65,820.50,Month-to-month,Fiber optic,Electronic check,Yes
C1006,62,89.10,5681.10,Two year,Fiber optic,Credit card (automatic),No
C1007,1,20.15,20.15,Month-to-month,No,Electronic check,No
C1008,72,103.70,7382.25,Two year,Fiber optic,Bank transfer (automatic),No
C1009,5,75.30,380.15,Month-to-month,Fiber optic,Electronic check,Yes
C1010,29,60.20,1745.80,One year,DSL,Credit card (automatic),No
C1011,0,45.00, ,Month-to-month,DSL,Mailed check,No
"""

df = pd.read_csv(StringIO(sample_csv))

df

,customerID,tenure,MonthlyCharges,TotalCharges,Contract,InternetService,PaymentMethod,Churn
0,C1001,2,70.35,140.70,Month-to-month,Fiber optic,Electronic check,Yes
1,C1002,34,56.95,1889.50,One year,DSL,Mailed check,No
2,C1003,2,53.85,108.15,Month-to-month,DSL,Mailed check,Yes
3,C1004,45,42.30,1840.75,One year,DSL,Bank transfer (automatic),No
4,C1005,8,99.65,820.50,Month-to-month,Fiber optic,Electronic check,Yes
5,C1006,62,89.10,5681.10,Two year,Fiber optic,Credit card (automatic),No
6,C1007,1,20.15,20.15,Month-to-month,No,Electronic check,No
7,C1008,72,103.70,7382.25,Two year,Fiber optic,Bank transfer (automatic),No
8,C1009,5,75.30,380.15,Month-to-month,Fiber optic,Electronic check,Yes
9,C1010,29,60.20,1745.80,One year,DSL,Credit card (automatic),No


In [4]:
# Inspect the Dataset

# Before preprocessing, inspect the structure of the DataFrame.

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())


Shape: (11, 8)

Columns:
['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'InternetService', 'PaymentMethod', 'Churn']

Missing Values:
customerID         0
tenure             0
MonthlyCharges     0
TotalCharges       0
Contract           0
InternetService    0
PaymentMethod      0
Churn              0
dtype: int64


In [5]:
# Encode the Target Variable

# The `Churn` column contains categorical values:

# - `Yes` → 1
# - `No` → 0

# This converts the target into numerical form required by Logistic Regression.

df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

print(df["Churn"].value_counts())

Churn
0    7
1    4
Name: count, dtype: int64


In [6]:
# # Clean TotalCharges

# `TotalCharges` contains a blank value for customer `C1011`.

# First, convert the column to numeric using:

# `pd.to_numeric(errors="coerce")`

# Invalid or blank values become `NaN`.

# Then replace the missing values with `0`.

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(0)

print(df[["customerID", "TotalCharges"]])

   customerID  TotalCharges
0       C1001        140.70
1       C1002       1889.50
2       C1003        108.15
3       C1004       1840.75
4       C1005        820.50
5       C1006       5681.10
6       C1007         20.15
7       C1008       7382.25
8       C1009        380.15
9       C1010       1745.80
10      C1011          0.00


In [7]:
# # Remove Customer Identifier

# `customerID` is an identifier rather than a meaningful predictive feature.

# Therefore, it is removed before creating the feature matrix.

df = df.drop("customerID", axis=1)

df.head()

,tenure,MonthlyCharges,TotalCharges,Contract,InternetService,PaymentMethod,Churn
0,2,70.35,140.70,Month-to-month,Fiber optic,Electronic check,1
1,34,56.95,1889.50,One year,DSL,Mailed check,0
2,2,53.85,108.15,Month-to-month,DSL,Mailed check,1
3,45,42.30,1840.75,One year,DSL,Bank transfer (automatic),0
4,8,99.65,820.50,Month-to-month,Fiber optic,Electronic check,1


In [8]:
# # Define Numerical and Categorical Columns

# Numerical columns:

# - tenure
# - MonthlyCharges
# - TotalCharges

# Categorical columns:

# - Contract
# - InternetService
# - PaymentMethod

num_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

cat_cols = [
    "Contract",
    "InternetService",
    "PaymentMethod"
]

print("Numerical Columns:", num_cols)
print("Categorical Columns:", cat_cols)

Numerical Columns: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical Columns: ['Contract', 'InternetService', 'PaymentMethod']


In [9]:
# # Separate Features and Target

# `X` contains the input features.

# `y` contains the target variable `Churn`.

X = df.drop("Churn", axis=1)

y = df["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11, 6)
y shape: (11,)


In [10]:
# # Train-Test Split

# We use:

# - Test size = 27%
# - Random state = 42
# - Stratification = y

# Stratification ensures that the proportion of churn and non-churn customers is preserved as much as possible in both datasets.

# For 11 records:

# - Training set = 8 records
# - Test set = 3 records
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.27,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 8
Testing rows: 3


# Create ColumnTransformer

Different types of features require different preprocessing.

### Numerical Features

Apply:

`StandardScaler()`

This standardizes the numerical features.

### Categorical Features

Apply:

`OneHotEncoder()`

This converts categorical values into numerical columns.

`handle_unknown="ignore"` ensures that an unseen category in the test data does not cause an error.

`Sparse_output=False` returns a dense NumPy array.

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            cat_cols
        )
    ]
)

In [12]:
# # Create Machine Learning Pipeline

# The pipeline combines:

# 1. Data preprocessing
# 2. Logistic Regression

# The Logistic Regression model uses:

# - `class_weight="balanced"`
# - `random_state=42`
# - `max_iter=1000`

# Using a Pipeline ensures that preprocessing is learned only from the training data when `.fit()` is called.
pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                random_state=42,
                max_iter=1000
            )
        )
    ]
)


In [13]:
# 

pipeline.fit(
    X_train,
    y_train
)

print("Pipeline training completed.")

Pipeline training completed.


In [14]:
# Make Predictions

# Use the trained pipeline to predict churn for the unseen test dataset.

y_pred = pipeline.predict(X_test)

print("Predictions:")
print(y_pred)

Predictions:
[1 1 1]


# Final Dataset Verification

After cleaning:

- `TotalCharges` blank value becomes `0.0`.
- `customerID` is removed.
- Numerical columns are scaled.
- Categorical columns are one-hot encoded.
- Churn is represented as 0 and 1.
- The training set contains 8 records.
- The test set contains 3 records.

# Conclusion

A complete customer churn prediction pipeline was successfully created using Pandas and Scikit-learn.

The pipeline combines data cleaning, numerical scaling, categorical encoding, and Logistic Regression into a single reusable workflow.

Using `ColumnTransformer` allows different preprocessing techniques to be applied to different feature types, while `Pipeline` ensures that preprocessing is performed correctly without data leakage.